# Deploy a Fine-Tuned Nova Model to Amazon Bedrock

This notebook demonstrates how to deploy a fine-tuned Amazon Nova model to
Amazon Bedrock using `BedrockModelBuilder`.

The workflow:
1. Retrieve a completed Nova SFT training job
2. Create a `BedrockModelBuilder` from the training job
3. Deploy to Bedrock — the builder automatically:
   - Detects the model as Nova
   - Reads the checkpoint URI from the training job manifest
   - Calls `CreateCustomModel` and polls until Active
   - Calls `CreateCustomModelDeployment` and polls until Active
4. Test inference
5. Clean up resources

**Prerequisites:**
- AWS credentials with SageMaker and Bedrock access
- `sagemaker-serve` package installed
- A completed Nova SFT training job
- An IAM role with Bedrock and SageMaker permissions

## Setup

In [ ]:
import os, json, time, random, boto3

REGION = "us-east-1"
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["SAGEMAKER_REGION"] = REGION

from sagemaker.core.helper.session_helper import get_execution_role
role_arn = get_execution_role()
print(f"Role: {role_arn}")

## Step 1: Retrieve the completed training job

Use an existing completed Nova SFT training job. Replace the job name with your own.

In [ ]:
from sagemaker.core.resources import TrainingJob

training_job = TrainingJob.get(training_job_name="nova-textgeneration-micro-sft-20251208154822")
print(f"Training job: {training_job.training_job_name}")
print(f"Status:       {training_job.training_job_status}")

## Step 2: Deploy to Bedrock with BedrockModelBuilder

The builder handles the full deployment flow:
- Fetches the model package from the training job
- Detects it as a Nova model
- Reads the checkpoint URI from the training output manifest
- Creates a Bedrock custom model and polls until Active
- Creates a deployment and polls until Active

In [ ]:
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

builder = BedrockModelBuilder(model=training_job)
print(f"Model package: {builder.model_package}")
print(f"S3 artifacts:  {builder.s3_model_artifacts}")

In [ ]:
rand = random.randint(1000, 9999)
custom_model_name = f"nova-e2e-{rand}-{int(time.time())}"
deployment_name = f"{custom_model_name}-dep"

print(f"Deploying as: {custom_model_name}")
print("This will poll for model creation and deployment — may take several minutes...")

response = builder.deploy(
    custom_model_name=custom_model_name,
    role_arn=role_arn,
    deployment_name=deployment_name,
)

deployment_arn = response.get("customModelDeploymentArn")
print(f"\nDeployment ARN: {deployment_arn}")
print("Deployment is Active and ready for inference.")

## Step 3: Test inference

Once the deployment is Active, invoke it via the Bedrock Runtime API.
Nova expects `content` as an array of objects with a `text` key.

In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

resp = bedrock_runtime.invoke_model(
    modelId=deployment_arn,
    contentType="application/json",
    body=json.dumps({
        "messages": [{"role": "user", "content": [{"text": "What is 7 + 7?"}]}]
    }),
)

result = json.loads(resp["body"].read())
print(f"Response: {json.dumps(result, indent=2)}")

## Step 4: Cleanup

Delete the deployment and custom model to avoid ongoing charges.

In [ ]:
bedrock = boto3.client("bedrock", region_name=REGION)

dep_info = bedrock.get_custom_model_deployment(
    customModelDeploymentIdentifier=deployment_arn
)
model_arn = dep_info.get("modelArn")

# Delete deployment first
try:
    bedrock.delete_custom_model_deployment(
        customModelDeploymentIdentifier=deployment_arn
    )
    print(f"Deleted deployment: {deployment_arn}")
except Exception as e:
    print(f"Failed to delete deployment: {e}")

# Then delete the custom model
try:
    bedrock.delete_custom_model(modelIdentifier=model_arn)
    print(f"Deleted custom model: {model_arn}")
except Exception as e:
    print(f"Failed to delete custom model: {e}")